In [ ]:
###################################################### run before using refined labels ###########################################################################################

import os
import shutil
from glob import glob
import SimpleITK as sitk
import numpy as np
from acvl_utils.morphology.morphology_helper import remove_all_but_largest_component


# ===========================================================
#   SLICE UTILITIES
# ===========================================================

def find_annotated_slices(label, axis):
    """Return sorted indices of slices containing label > 0."""
    if axis == "axial":
        reduce_axes = (1, 2)
    elif axis == "coronal":
        reduce_axes = (0, 2)
    elif axis == "sagittal":
        reduce_axes = (0, 1)
    else:
        raise ValueError("Invalid axis")

    return np.where(np.any(label > 0, axis=reduce_axes))[0]


# ===========================================================
#   GAP-AWARE INTERPOLATION
# ===========================================================

def interpolate_gap(lbl, k0, k1):
    """Interpolate slices strictly between k0 and k1."""
    for k in range(k0 + 1, k1):
        w = (k - k0) / float(k1 - k0)
        lbl[k] = ((1 - w) * lbl[k0] + w * lbl[k1]) > 0
    return lbl


def interpolate_missing_slices(label, axis="axial", max_gap=5):
    """
    Interpolate ONLY small gaps (<= max_gap) between annotated slices.
    """
    lbl = label.copy()

    # Reorient so slice dimension = 0
    if axis == "coronal":
        lbl = np.transpose(lbl, (1, 0, 2))
    elif axis == "sagittal":
        lbl = np.transpose(lbl, (2, 1, 0))

    annotated = find_annotated_slices(lbl, "axial")

    if len(annotated) < 2:
        return label

    for i in range(len(annotated) - 1):
        k0, k1 = annotated[i], annotated[i + 1]
        gap = k1 - k0 - 1

        if 1 <= gap <= max_gap:
            lbl = interpolate_gap(lbl, k0, k1)

    # Restore orientation
    if axis == "coronal":
        lbl = np.transpose(lbl, (1, 0, 2))
    elif axis == "sagittal":
        lbl = np.transpose(lbl, (2, 1, 0))

    return lbl.astype(label.dtype)


# ===========================================================
#   RESAMPLING
# ===========================================================

def resample_label_to_image(label_path, image_ref_path):
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))

    return sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID(),
    )


# ===========================================================
#   PATHS
# ===========================================================

label_dir = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/original"
image_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr"

out_image_dir = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/imagesTr"
out_label_dir = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/predictionsTs"

os.makedirs(out_image_dir, exist_ok=True)
os.makedirs(out_label_dir, exist_ok=True)

label_files = sorted(glob(os.path.join(label_dir, "*.nii.gz")))
print(f"Found {len(label_files)} label files")


# ===========================================================
#   MAIN PIPELINE
# ===========================================================

MAX_GAP = 4

for label_path in label_files:
    label_name = os.path.basename(label_path)
    uid = label_name.split("_")[0]
    img_path = os.path.join(image_dir, label_name)

    if not os.path.exists(img_path):
        print(f"WARNING: No matching image for {label_name}")
        continue

    # Copy image
    new_img_name = f"{uid}_0000.nii.gz"
    shutil.copy2(img_path, os.path.join(out_image_dir, new_img_name))

    # Load + resample label
    label_resampled = resample_label_to_image(label_path, img_path)
    label_np = sitk.GetArrayFromImage(label_resampled).astype(np.uint8)
    label_np[label_np == 2] = 0

    # --- Report gaps BEFORE interpolation ---
    annotated = find_annotated_slices(label_np, "axial")
    print(f"{uid}: annotated slices = {annotated}")

    for i in range(len(annotated) - 1):
        gap = annotated[i + 1] - annotated[i] - 1
        if gap > 0:
            print(f"  gap of {gap} slices between {annotated[i]} and {annotated[i+1]}")

    # --- Interpolate small gaps FIRST ---
    """
    mask_interp = interpolate_missing_slices(
        label_np,
        axis="axial",
        max_gap=MAX_GAP,
    )
    """
    # --- NOW keep only largest component ---
    mask_final = remove_all_but_largest_component(label_np).astype(np.uint8)

    if not np.array_equal(mask_final, label_np):
        print(f"✓ Updated mask for {uid}")
    else:
        print(f"Keeping original mask for {uid}")

    # Save label
    final_label_sitk = sitk.GetImageFromArray(mask_final)
    final_label_sitk.CopyInformation(label_resampled)

    new_label_name = f"{uid}.nii.gz"
    sitk.WriteImage(final_label_sitk, os.path.join(out_label_dir, new_label_name))

    print(f"Processed {uid}: Image={new_img_name}, Label={new_label_name}")

print("Done!")


In [ ]:
############################################################## sanity check refined labels ###################################################################

import os
from glob import glob
import SimpleITK as sitk
import numpy as np
from scipy.ndimage import label as cc_label
from acvl_utils.morphology.morphology_helper import remove_all_but_largest_component
from datetime import datetime

# ===========================================================
#   SLICE UTILITIES
# ===========================================================

def find_annotated_slices(label, axis="axial"):
    """Return sorted indices of slices containing label > 0."""
    if axis == "axial":
        reduce_axes = (1, 2)
    elif axis == "coronal":
        reduce_axes = (0, 2)
    elif axis == "sagittal":
        reduce_axes = (0, 1)
    else:
        raise ValueError("Invalid axis")

    return np.where(np.any(label > 0, axis=reduce_axes))[0]


# ===========================================================
#   PATHS
# ===========================================================

label_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
label_files = sorted(glob(os.path.join(label_dir, "*.nii.gz")))

log_dir = "./qc_logs"
os.makedirs(log_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(log_dir, f"label_qc_report_{timestamp}.txt")

MAX_GAP = 4

print(f"Found {len(label_files)} label files")
print(f"Logs will be saved to: {log_path}")

# ===========================================================
#   DATASET-LEVEL CONTAINERS
# ===========================================================

cases_multi_label = []
cases_small_gaps = []
cases_large_gaps = []
cases_parts_removed = []

# ===========================================================
#   LOGGING UTILITY
# ===========================================================

def log(msg):
    print(msg)
    with open(log_path, "a") as f:
        f.write(msg + "\n")


# ===========================================================
#   MAIN INSPECTION LOOP
# ===========================================================

with open(log_path, "w") as f:
    f.write(f"Label QC Report - {timestamp}\n")
    f.write("=" * 60 + "\n\n")

for label_path in label_files:
    name = os.path.basename(label_path)
    uid = name.split("_")[0]

    label_sitk = sitk.ReadImage(label_path)
    label_np = sitk.GetArrayFromImage(label_sitk)

    log(f"\n=== CASE {uid} ===")

    # -------------------------------------------------------
    # 1. UNIQUE LABEL VALUES
    # -------------------------------------------------------
    unique_vals = np.unique(label_np)
    log(f"Unique label values: {unique_vals.tolist()}")

    if len(unique_vals) > 2:
        cases_multi_label.append(uid)
        log("  ⚠ More than one foreground label present")

    # -------------------------------------------------------
    # 2. GAP ANALYSIS (AXIAL)
    # -------------------------------------------------------
    annotated = find_annotated_slices(label_np, axis="axial")
    log(f"Annotated axial slices: {annotated.tolist()}")

    has_small_gap = False
    has_large_gap = False

    for i in range(len(annotated) - 1):
        gap = annotated[i + 1] - annotated[i] - 1
        if gap > 0:
            if gap <= MAX_GAP:
                has_small_gap = True
                log(f"  ⚠ Small gap ({gap} slices) between {annotated[i]} and {annotated[i+1]}")
            else:
                has_large_gap = True
                log(f"  ❌ Large gap ({gap} slices) between {annotated[i]} and {annotated[i+1]}")

    if has_small_gap:
        cases_small_gaps.append(uid)

    if has_large_gap:
        cases_large_gaps.append(uid)

    # -------------------------------------------------------
    # 3. CONNECTED COMPONENTS (ACTUAL REMOVAL CHECK)
    # -------------------------------------------------------
    binary = (label_np > 0).astype(np.uint8)

    _, num_components_before = cc_label(binary)

    largest_only = remove_all_but_largest_component(binary)

    voxels_before = binary.sum()
    voxels_after = largest_only.sum()
    removed_voxels = voxels_before - voxels_after

    log(f"Connected components (before): {num_components_before}")
    log(f"Foreground voxels before: {voxels_before}")
    log(f"Foreground voxels after : {voxels_after}")

    if removed_voxels > 0:
        cases_parts_removed.append(uid)
        log(
            f"  ❌ remove_all_but_largest_component removed "
            f"{removed_voxels} voxels"
        )
    else:
        log("  ✓ No components removed")

# ===========================================================
#   FINAL SUMMARY
# ===========================================================

log("\n" + "=" * 60)
log("FINAL SUMMARY")
log("=" * 60)

log(f"Cases with >1 unique label value : {len(cases_multi_label)}")
log(f"Cases with small gaps (≤ {MAX_GAP}) : {len(cases_small_gaps)}")
log(f"Cases with large gaps (> {MAX_GAP}) : {len(cases_large_gaps)}")
log(f"Cases where parts were removed    : {len(cases_parts_removed)}")

log("\nIDs with multiple labels:")
log(cases_multi_label)

log("\nIDs with small gaps:")
log(cases_small_gaps)

log("\nIDs with large gaps:")
log(cases_large_gaps)

log("\nIDs where components were removed:")
log(cases_parts_removed)

log("\nInspection complete.")


In [ ]:
################################################################# retrieve test samples #########################################################

import os
import shutil

src_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/labelsTr"
dst_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/labelsTs"

# List of filenames to move
files_to_move = [
    "120.nii.gz", "15.nii.gz", "206.nii.gz", "300.nii.gz", "339.nii.gz", "52.nii.gz",
    "564.nii.gz", "646.nii.gz", "724.nii.gz", "821.nii.gz", "151.nii.gz", "18.nii.gz",
    "29.nii.gz", "330.nii.gz", "426.nii.gz", "530.nii.gz", "571.nii.gz", "718.nii.gz",
    "813.nii.gz", "86.nii.gz"
]

# Create destination folder if missing
os.makedirs(dst_dir, exist_ok=True)

for fname in files_to_move:
    src_path = os.path.join(src_dir, fname)
    dst_path = os.path.join(dst_dir, fname)

    if os.path.exists(src_path):
        print(f"Moving {fname} ...")
        shutil.move(src_path, dst_path)
    else:
        print(f"NOT FOUND: {fname}")


In [ ]:
############################################################################# check annotation volumes after resampling them  ######################################################################################################
import os
import numpy as np

# === CONFIG ===
LABEL_DIR = "/data/colon_cancer/CC_Detection/raw_data/Dataset107_CC/labelsTr"
EXTENSIONS = [".npy", ".nii", ".nii.gz"]
THRESHOLD = 20            # voxel threshold for tiny annotations
SLICE_AXIS = 2            # 0=sagittal, 1=coronal, 2=axial

try:
    import nibabel as nib
    has_nibabel = True
except ImportError:
    has_nibabel = False


def load_label(filepath):
    """Load a label file (supports .npy or .nii/.nii.gz)."""
    ext = os.path.splitext(filepath)[1]
    if ext == ".npy":
        return np.load(filepath)
    elif ext in [".nii", ".gz"] and has_nibabel:
        return nib.load(filepath).get_fdata()
    else:
        raise ValueError(f"Unsupported format or missing nibabel: {filepath}")


def find_empty_slices_inside_mask(label, axis=2):
    """
    Detect empty slices inside the segmented region along a given axis.

    Returns:
        has_gap (bool)
        empty_slices (list of slice indices)
    """
    mask = label > 0

    # Sum foreground per slice
    slice_sums = np.sum(
        mask,
        axis=tuple(i for i in range(mask.ndim) if i != axis)
    )

    fg_slices = np.where(slice_sums > 0)[0]

    if len(fg_slices) == 0:
        return False, []

    first, last = fg_slices[0], fg_slices[-1]

    empty_slices = [
        i for i in range(first + 1, last)
        if slice_sums[i] == 0
    ]

    return len(empty_slices) > 0, empty_slices


def check_labels():
    label_files = [
        os.path.join(LABEL_DIR, f)
        for f in os.listdir(LABEL_DIR)
        if any(f.endswith(ext) for ext in EXTENSIONS)
    ]

    if not label_files:
        print(f"No label files found in {LABEL_DIR}")
        return

    volumes = []
    gap_reports = []

    print(f"Found {len(label_files)} label files. Computing volumes...\n")

    for path in label_files:
        try:
            label = load_label(path)
            volume = int(np.sum(label > 0))

            has_gap, empty_slices = find_empty_slices_inside_mask(
                label, axis=SLICE_AXIS
            )

            volumes.append((path, volume))
            gap_reports.append((path, has_gap, empty_slices))

        except Exception as e:
            print(f"⚠️ Could not load {path}: {e}")

    # Sort by volume
    volumes.sort(key=lambda x: x[1])

    print("=== Label volumes (sorted) ===")
    for path, vol in volumes:
        print(f"{os.path.basename(path):40s}  -> {vol}")

    print("\n=== Labels with empty or tiny annotations ===")
    for path, vol in volumes:
        if vol == 0:
            print(f"🚫 EMPTY: {os.path.basename(path)}")
        elif vol < THRESHOLD:
            print(f"⚠️ Low volume ({vol} voxels): {os.path.basename(path)}")

    print("\n=== Labels with segmentation gaps (empty slices inside ROI) ===")
    any_gaps = False
    for path, has_gap, empty_slices in gap_reports:
        if has_gap:
            any_gaps = True
            print(
                f"🕳️ GAP: {os.path.basename(path)} | "
                f"{len(empty_slices)} empty slices | "
                f"indices: {empty_slices}"
            )

    if not any_gaps:
        print("✅ No segmentation gaps detected.")


if __name__ == "__main__":
    check_labels()


In [ ]:
########################################################################################## extract top 50-loss from validation predictions ###################################################################################### 
import json
from pathlib import Path
import shutil


# List of JSON summary files
json_files = [
    "/data/colon_cancer/nnUNet_results/Dataset102_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation/summary.json",
    "/data/colon_cancer/nnUNet_results/Dataset102_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_1/validation/summary.json",
    "/data/colon_cancer/nnUNet_results/Dataset102_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_2/validation/summary.json",
    "/data/colon_cancer/nnUNet_results/Dataset102_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_3/validation/summary.json",
    "/data/colon_cancer/nnUNet_results/Dataset102_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_4/validation/summary.json",
]

cases = []

for json_file in json_files:
    try:
        with open(json_file, "r") as f:
            data = json.load(f)
        for entry in data.get("metric_per_case", []):
            metrics = entry.get("metrics", {})
            class1 = metrics.get("1", {})
            dice = class1.get("Dice", None)
            pred_file = entry.get("prediction_file", None)
            if dice is not None and pred_file is not None:
                cases.append((dice, pred_file))
            print(f"done for {pred_file}")
    except Exception as e:
        print(f"⚠️ Could not parse {json_file}: {e}")

# Sort by Dice ascending → worst first
cases_sorted = sorted(cases, key=lambda x: x[0])

# Take the worst 50
worst_50 = cases_sorted[:50]

# Extract prediction file paths
worst_50_paths = [p for _, p in worst_50]

print("Worst 50 prediction files:")
for p in worst_50_paths:
    print(p)

############################################################################################ prepare for next refining round #########################################################################################################

# Target folder to copy all files
target_folder = Path("/data/colon_cancer/nnUNet_worst_predictions")
target_folder.mkdir(parents=True, exist_ok=True)

# Copy files
for file_path in worst_50_paths:
    src = Path(file_path)
    if src.exists():
        shutil.copy2(src, target_folder / src.name)
    else:
        print(f"⚠️ File does not exist: {src}")

print(f"Copied {len(worst_50_paths)} files to {target_folder}")

# Folder containing the worst predictions
source_dir = Path("/data/colon_cancer/nnUNet_worst_predictions")

# Target folder
target_dir = Path("/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original")
target_dir.mkdir(parents=True, exist_ok=True)

# Iterate over all NIfTI files in the source folder
for src in source_dir.glob("*.nii.gz"):
    # Extract UID from filename by stripping leading zeros
    uid = str(int(src.name.replace(".nii.gz", "")))  # converts "025" -> "25", "003" -> "3"
    print(uid)
    # Destination path
    dst = target_dir / f"{uid}_0000.nii.gz"
    print(dst)
   
    # Move and overwrite if exists
    shutil.move(str(src), str(dst))
    print(f"Moved: {src} -> {dst}")

print("All worst prediction files have been moved and renamed with UID format.")


# Source folder with the "final" files
final_dir = Path("/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final")

# Destination folder where matching files will be moved
destination_dir = Path("/data/colon_cancer/nnUNet_worst_predictions_final")
destination_dir.mkdir(parents=True, exist_ok=True)

# Extract UIDs from worst prediction filenames
uids = set()
for file_path in worst_50_paths:
    uid = str(int(Path(file_path).name.replace(".nii.gz", "")))  
    uids.add(uid)

# Move matching files from final_dir to destination_dir
for src_file in final_dir.glob("*.nii.gz"):
    # Extract UID from file, e.g., "25_0000.nii.gz" -> "25"
    uid_part = src_file.stem.split("_")[0]
    if uid_part in uids:
        dst_file = destination_dir / src_file.name
        shutil.move(str(src_file), str(dst_file))
        print(f"Moved: {src_file} -> {dst_file}")

print("All matching worst prediction files have been moved.")

In [ ]:
######################################################  merge GT to predictions for selected samples ###########################################################################################################

import os
import nibabel as nib
import numpy as np

def edit_labels(
    folder_path: str,
    target_files: list,
    overwrite: bool = True,
    output_dir: str = None,
):
    """
    Replace labels {1, 2} -> 1 in selected NIfTI files.

    Args:
        folder_path (str): Path to the folder containing label .nii.gz files.
        target_files (list): List of filenames to modify (e.g. ["1_0000.nii.gz"]).
        overwrite (bool): If True, modifies files in place.
        output_dir (str): If not overwriting, save modified labels to this folder.
    """

    # Create output directory if needed
    if not overwrite:
        if output_dir is None:
            raise ValueError("output_dir must be specified when overwrite=False")
        os.makedirs(output_dir, exist_ok=True)

    # Crawl folder recursively
    for root, _, files in os.walk(folder_path):
        for fname in files:
            if fname.endswith(".nii.gz") and fname in target_files:

                full_path = os.path.join(root, fname)
                print(f"Editing: {full_path}")

                # Load label volume
                img = nib.load(full_path)
                data = img.get_fdata()

                # Replace 1 and 2 with 1
                data[data>0] = 1

                # Save result
                new_img = nib.Nifti1Image(data.astype(np.uint8), img.affine, img.header)

                if overwrite:
                    nib.save(new_img, full_path)
                else:
                    save_path = os.path.join(output_dir, fname)
                    nib.save(new_img, save_path)
                    print(f"Saved edited file to: {save_path}")

    print("Done!")


# -------------------------
# Example usage:
# -------------------------

folder = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr"

# List of label filenames you want to edit
to_edit = [
    '1_0000.nii.gz', '119_0000.nii.gz', '125_0000.nii.gz', '126_0000.nii.gz',
    '142_0000.nii.gz', '251_0000.nii.gz', '255_0000.nii.gz', '274_0000.nii.gz',
    '349_0000.nii.gz', '371_0000.nii.gz', '382_0000.nii.gz', '404_0000.nii.gz',
    '458_0000.nii.gz', '514_0000.nii.gz', '515_0000.nii.gz', '521_0000.nii.gz',
    '531_0000.nii.gz', '532_0000.nii.gz', '537_0000.nii.gz', '547_0000.nii.gz',
    '556_0000.nii.gz', '561_0000.nii.gz', '570_0000.nii.gz', '577_0000.nii.gz',
    '582_0000.nii.gz', '584_0000.nii.gz', '587_0000.nii.gz', '597_0000.nii.gz',
    '674_0000.nii.gz', '789_0000.nii.gz',
    '141_0000.nii.gz', '198_0000.nii.gz', '360_0000.nii.gz', '367_0000.nii.gz',
    '400_0000.nii.gz', '533_0000.nii.gz', '545_0000.nii.gz', '585_0000.nii.gz',
    '680_0000.nii.gz', '703_0000.nii.gz'
]



edit_labels(folder_path=folder, target_files=to_edit, overwrite=True)



import os
from pathlib import Path
import numpy as np
import nibabel as nib

# --- Paths ---
original_labels_dir = Path("/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original")
new_labels_dir = Path("/data/colon_cancer/Task101_Colon/raw_splitted/labelsTr")
merged_labels_dir = Path("/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original")
merged_labels_dir.mkdir(parents=True, exist_ok=True)

# --- List of IDs / files ---
file_ids =  [660, 11, 359, 696, 75, 222, 310, 363, 175, 752, 658, 693, 682, 505, 177]

for uid in file_ids:
    # Original label file
    orig_file_candidates = [
        f"{uid}_0000.nii.gz"
    ]
    orig_file = None
    for f in orig_file_candidates:
        f_path = original_labels_dir / f
        if f_path.exists():
            orig_file = f_path
            break
    if orig_file is None:
        print(f"Original label not found for UID {uid}")
        continue

    # New label file
    new_file_candidates = [
         f"{uid}.nii.gz"
    ]
    new_file = None
    for f in new_file_candidates:
        f_path = new_labels_dir / f
        if f_path.exists():
            new_file = f_path
            break
    if new_file is None:
        print(f"New label not found for UID {uid}")
        continue

    # --- Load NIfTI files ---
    orig_nii = nib.load(orig_file)
    orig_data = orig_nii.get_fdata().astype(np.uint8)

    new_nii = nib.load(new_file)
    new_data = new_nii.get_fdata().astype(np.uint8)

    print(orig_data.shape, orig_data.sum())
    print(new_data.shape, new_data.sum())
    # --- Merge labels ---
    # Original 1 stays 1, new 1 becomes 2, 0 stays 0
    merged_data = orig_data.copy()
    merged_data[new_data == 1] = 2
    print(merged_data.shape, merged_data.sum())
    # --- Save merged file ---
    merged_nii = nib.Nifti1Image(merged_data, affine=orig_nii.affine, header=orig_nii.header)
    out_path = merged_labels_dir / f"{uid}_0000.nii.gz"
    nib.save(merged_nii, out_path)
    print(f"Merged labels saved for UID {uid} → {out_path}")



In [ ]:
#############################################  merge GT to predictions for all samples  ###########################################################################################################


import os
import nibabel as nib
import numpy as np
import shutil

def merge_predictions_and_gt(
    preds_dir,
    gt_dir,
    merged_out_dir,
    remove_dir,
    processed_list_path="processed_files.txt"
):
    os.makedirs(merged_out_dir, exist_ok=True)

    # Load already processed files
    if os.path.exists(processed_list_path):
        with open(processed_list_path, "r") as f:
            processed = set(line.strip() for line in f.readlines())
    else:
        processed = set()

    new_processed = []

    pred_files = [f for f in os.listdir(preds_dir) if f.endswith(".nii.gz")]

    for pred_file in pred_files:
        base = pred_file.replace(".nii.gz", "")

        if base in processed:
            print(f"[SKIP] Already processed: {base}")
            continue

        pred_path = os.path.join(preds_dir, pred_file)
        gt_path = os.path.join(gt_dir, pred_file)

        if not os.path.exists(gt_path):
            print(f"[WARN] No GT file found for {pred_file}, skipping.")
            continue

        # Load images
        pred_img = nib.load(pred_path) 
        gt_img = nib.load(gt_path)

        pred_data = pred_img.get_fdata()
        gt_data = gt_img.get_fdata()

        # Create merged mask
        merged_mask = np.zeros_like(pred_data, dtype=np.uint8)

        merged_mask[pred_data > 0] = 1  # prediction label = 1
        merged_mask[gt_data > 0] = 2    # GT label = 2

        # Save merged
        file_0000 = base + "_0000.nii.gz"
        out_path = os.path.join(merged_out_dir, file_0000)
        merged_nifti = nib.Nifti1Image(merged_mask, pred_img.affine, pred_img.header)
        nib.save(merged_nifti, out_path)

        print(f"[SAVED] {out_path}")

        # Remove corresponding *_0000.nii.gz in remove_dir
        
        remove_path = os.path.join(remove_dir, file_0000)

        if os.path.exists(remove_path):
            os.remove(remove_path)
            print(f"[REMOVED] {remove_path}")

        new_processed.append(base)
        

    # Update processed list
    with open(processed_list_path, "a") as f:
        for name in new_processed:
            f.write(name + "\n")

    print("Done.")


if __name__ == "__main__":
    merge_predictions_and_gt(
        preds_dir="/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTr",
        gt_dir="/data/colon_cancer/Task101_Colon/raw_splitted/labelsTr",
        merged_out_dir="/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original",
        remove_dir="/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final",
        processed_list_path="/data/colon_cancer/Task101_Colon/processed_files.txt"
    )


In [ ]:
################################################# test functions for cleaning predictions ##########################################################
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import cc3d

def keep_components_touching_colon(pred_mask, colon_mask, connectivity=26):
    """
    Keeps only connected components in pred_mask that intersect the colon mask.
    Prints the slice locations of kept and discarded components.
    """
    labeled, num = cc3d.connected_components(pred_mask, return_N=True, connectivity=connectivity)
    colon_mask_bool = colon_mask > 0
    keep_ids = []
    discarded_ids = []

    for comp_id in range(1, num + 1):
        component_voxels = (labeled == comp_id)
        slices_with_component = np.where(component_voxels.sum(axis=(0,1)) > 0)[0]
        if np.any(component_voxels & colon_mask_bool):
            keep_ids.append(comp_id)
            print(f"Component {comp_id} kept, slices: {slices_with_component}")
        else:
            discarded_ids.append(comp_id)
            print(f"Component {comp_id} discarded, slices: {slices_with_component}")

    print(f"Keeping {len(keep_ids)} / {num} components that touch colon")
    cleaned = np.isin(labeled, keep_ids).astype(np.uint8)
    return cleaned


def load_and_clean(uid, gt_path=None, window_center=None, window_width=None):
    """
    uid: patient/sample ID
    gt_path: optional path to ground truth mask
    window_center / window_width: optional CT windowing
    """
    # --------------------------
    # Paths
    # --------------------------
    pred_path = f"/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs/{uid}.nii.gz"
    colon_path = f"/data/colon_cancer/totalseg/total/{uid}.nii.gz"
    img_path   = f"/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTs/{uid}_0000.nii.gz"
    gt_path = f"/data/colon_cancer/Task101_Colon/raw_splitted/labelsTs/{uid}.nii.gz"

    # --------------------------
    # Load images
    # --------------------------
    pred = nib.load(pred_path).get_fdata().astype(np.uint8)
    colon = nib.load(colon_path).get_fdata().astype(np.uint8)
    image = nib.load(img_path).get_fdata()
    
    # Apply windowing if specified
    if window_center is not None and window_width is not None:
        lower = window_center - window_width // 2
        upper = window_center + window_width // 2
        image = np.clip(image, lower, upper)
        image = (image - lower) / window_width  # normalize to 0-1

    # Load GT mask if provided
    if gt_path:
        gt_mask = nib.load(gt_path).get_fdata().astype(np.uint8)
    else:
        gt_mask = None

    # --------------------------
    # Clean mask
    # --------------------------
    cleaned = keep_components_touching_colon(pred, colon)

    # --------------------------
    # Slices with annotations
    # --------------------------
    slices_with_annotations = np.where(cleaned.sum(axis=(0,1)) > 0)[0]
    print(f"Slices with cleaned mask annotations: {slices_with_annotations}")

    # --------------------------
    # Visualization
    # --------------------------
    def display_slice(idx):
        fig, ax = plt.subplots(1, 5 if gt_mask is not None else 4, figsize=(20, 6))

        base = image[:, :, idx]

        # 0 – Image only
        ax[0].imshow(base, cmap="gray")
        ax[0].set_title(f"Image (slice {idx})")
        ax[0].axis("off")

        # 1 – Image + original prediction overlay
        ax[1].imshow(base, cmap="gray")
        ax[1].imshow(pred[:, :, idx], cmap="Reds", alpha=0.4)
        ax[1].set_title("Original prediction overlay")
        ax[1].axis("off")

        # 2 – Image + colon overlay
        ax[2].imshow(base, cmap="gray")
        ax[2].imshow(colon[:, :, idx], cmap="Blues", alpha=0.4)
        ax[2].set_title("Colon mask overlay")
        ax[2].axis("off")

        # 3 – Image + cleaned mask overlay
        ax[3].imshow(base, cmap="gray")
        ax[3].imshow(cleaned[:, :, idx], cmap="Greens", alpha=0.4)
        ax[3].set_title("Cleaned mask overlay")
        ax[3].axis("off")

        # 4 – Image + GT mask overlay (optional)
        if gt_mask is not None:
            ax[4].imshow(base, cmap="gray")
            ax[4].imshow(gt_mask[:, :, idx], cmap="Purples", alpha=0.4)
            ax[4].set_title("GT mask overlay")
            ax[4].axis("off")

        plt.tight_layout()
        plt.show()

    interact(
        display_slice,
        idx=widgets.IntSlider(min=0, max=image.shape[2]-1, step=1, value=image.shape[2] // 2)
    )

    print(f"Total voxels in cleaned mask: {cleaned.sum()}")


# Example
#113,10,123,136,146
#1, 10, 113,  "18.nii.gz", 29 339 426 52 

load_and_clean(724,window_center=50,window_width=350)


In [ ]:
################# merge refined labels with old labels #########################
import os
import shutil
from glob import glob

# === SOURCE FOLDERS ===
src_image_dirs = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr"

src_label_dirs = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr"

# === DESTINATION FOLDERS ===
dst_images = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/imagesTr"
dst_labels = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/labelsTr"


# === Copy helper ===
def copy_missing(src_folder, dst_folder):
    src_files = sorted(glob(os.path.join(src_folder, "*")))
    missing_count = 0

    for src_file in src_files:
        fname = os.path.basename(src_file)
        dst_file = os.path.join(dst_folder, fname)
        if not os.path.exists(dst_file):
            shutil.copy2(src_file, dst_file)
            print(f"➕ Copied: {fname}")
            missing_count += 1
        

    print(f"\n→ Done. Added {missing_count} missing files from {src_folder}.\n")


copy_missing(src_image_dirs, dst_images)


copy_missing(src_label_dirs, dst_labels)

print("All folders synced.")


In [ ]:
################################## prepare for refining rounds with specific samples ############################################################
import os
import shutil

# --- INPUTS ---

#uids = [577, 105, 116, 125, 25, 294, 328, 366, 394, 577, 693, 777, 806]  
uids =  [
     33, 37, 38, 42, 46, 50, 56, 62, 73, 79, 92, 95, 99,
    115, 122, 136, 145, 154, 158, 163, 166, 168, 174, 177, 188, 200,
    222, 223, 250, 258, 284, 289, 302, 319, 354, 355, 381, 387,
    417, 437, 439, 446, 454, 461, 465, 481, 491, 508, 519,
    553, 555, 558, 569, 580, 583, 599, 626, 645,
    658, 679, 697, 714, 723, 760, 773, 805
]


dst_A = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"  # folder where you move files IN the list
dst_B = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"  # folder where you move files NOT IN the list
dst_C = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr"

src_folder_A = "/data/colon_cancer/Task101_Colon/raw_splitted/labels_final_extracted"
src_folder_B = "/data/colon_cancer/Task101_Colon/raw_splitted/labels_original_extracted"
src_folder_C = "/data/colon_cancer/Task101_Colon/raw_splitted/images_extracted"


# Create destination folders if they don't exist
os.makedirs(dst_A, exist_ok=True)
os.makedirs(dst_B, exist_ok=True)
os.makedirs(dst_C, exist_ok=True)


# Helper function to extract UID from filename
def extract_uid(filename):
    """Extract UID before the first underscore, e.g.  '356_0000.nii.gz' -> 356."""
    try:
        return int(filename.split("_")[0])
    except:
        return None


# --- MOVE FILES FROM FOLDER A (UID in list) ---

for filename in os.listdir(src_folder_A):
    if filename.endswith(".nii.gz"):
        uid = extract_uid(filename)
        if uid is None or "old" in filename:
            continue

        src_path = os.path.join(src_folder_A, filename)

        # UID is in the list → move to dst_in_list
        if uid in uids:
            dst_path = os.path.join(dst_A, filename)
            print(f"Moving {filename} → final labels")
            shutil.move(src_path, dst_path)

"""
# --- MOVE FILES FROM FOLDER B and C (UID not in list) ---

for filename in os.listdir(src_folder_B):
    if filename.endswith(".nii.gz"):
        uid = extract_uid(filename)
        if uid is None or "old" in filename:
            continue

        src_path_B = os.path.join(src_folder_B, filename)
        src_path_C = os.path.join(src_folder_C, filename)

        # UID not in list → move to dst_not_in_list
        if uid in uids:
            dst_path_B = os.path.join(dst_B, filename)
            dst_path_C = os.path.join(dst_C, filename)
            print(f"Moving {filename} → Image and Labels")
            shutil.move(src_path_B, dst_path_B)
            shutil.move(src_path_C, dst_path_C)
"""
print("Done!")


In [ ]:
############################################# move processed images ###############################################
import os
import shutil

source_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"      # Folder A (contains .nii.gz files)
reference_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"  # Folder B
output_dir = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/original"

os.makedirs(output_dir, exist_ok=True)

# Get base filenames (without extensions) from reference folder
ref_names = {
    os.path.splitext(os.path.splitext(f)[0])[0]
    for f in os.listdir(reference_dir)
}

# Copy matching .nii.gz files
for f in os.listdir(source_dir):
    if f.endswith(".nii.gz"):
        base = os.path.splitext(os.path.splitext(f)[0])[0]
        if base in ref_names:
            shutil.copy(
                os.path.join(source_dir, f),
                os.path.join(output_dir, f)
            )

print("Done.")


In [ ]:
import os
import csv

root_dir = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/imagesTs"
output_csv = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/labels.csv"

rows = []

for root, _, files in os.walk(root_dir):
    for f in files:
        if f.endswith("_0000.nii.gz"):
            uid = f.replace("_0000.nii.gz", "")
            rows.append([uid, 0])

# Write CSV
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["UID", "target"])
    writer.writerows(rows)

print(f"Saved {len(rows)} entries to {output_csv}")


In [ ]:
import os
import shutil

folder_images = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/imagesTs"
folder_preds = "/data/colon_cancer/totalseg/outputs_cc2stage/total"
output_folder = "/data/colon_cancer/CC_Detection/raw_data/stage_2_cc/imagesTs_no_preds"

os.makedirs(output_folder, exist_ok=True)

# Get UIDs that have predictions
pred_uids = {
    os.path.splitext(os.path.splitext(f)[0])[0]
    for f in os.listdir(folder_preds)
    if f.endswith(".nii.gz")
}

# Copy images without predictions
for f in os.listdir(folder_images):
    if f.endswith("_0000.nii.gz"):
        uid = f.replace("_0000.nii.gz", "")
        if uid not in pred_uids:
            src = os.path.join(folder_images, f)
            dst = os.path.join(output_folder, f)
            print(f"Copying {src}")
            shutil.copy(src, dst)

print("Done.")


In [ ]:
CUDA_VISIBLE_DEVICES=0 nohup nnUNetv2_predict -d 109 -i /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs -o /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final_predictionsTs -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_fullres -p nnUNetResEncUNetLPlans -npp 1 -nps 1 
CUDA_VISIBLE_DEVICES=0 nohup nnUNetv2_predict -d 109 -i /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr -o /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final_predictionsTr -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_fullres -p nnUNetResEncUNetLPlans -npp 1 -nps 1 

In [ ]:
import os
import nibabel as nib
import numpy as np
from pathlib import Path

def keep_specific_labels(input_folder, output_folder, keep_labels=[1, 2, 3], target_mapping=None):
    """
    Crawls through .nii.gz files, keeps only specified labels, optionally maps them
    to new values, and sets everything else to 0.
    
    Parameters:
    -----------
    input_folder : str
        Path to folder containing input mask files
    output_folder : str
        Path to folder where processed masks will be saved
    keep_labels : list
        List of label values to keep from the original masks
    target_mapping : dict, optional
        Dictionary mapping original labels to new values.
        Example: {1: 1, 2: 1, 3: 2} maps labels 1 and 2 to 1, and label 3 to 2.
        If None, original labels are preserved.
    """
    # Create output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    # Filter for NIfTI files
    mask_files = [f for f in os.listdir(input_folder) if f.endswith('.nii.gz')]

    if not mask_files:
        print("No .nii.gz files found in the input folder.")
        return

    for filename in mask_files:
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # Load the NIfTI image
        img = nib.load(input_path)
        data = img.get_fdata()
        header = img.header
        affine = img.affine

        # Create new mask initialized with zeros
        new_data = np.zeros_like(data, dtype=np.uint8)

        if target_mapping is None:
            # Keep original labels, only for those in keep_labels
            for label in keep_labels:
                new_data[data == label] = label
        else:
            # Map specified labels to new values
            for original_label, new_label in target_mapping.items():
                if original_label in keep_labels:
                    new_data[data == original_label] = new_label

        # Create a new NIfTI object with original affine and header
        new_img = nib.Nifti1Image(new_data, affine, header)

        # Save the processed mask
        nib.save(new_img, output_path)
        print(f"Processed: {filename}")

# --- Configuration ---
input_dir = "/data/colon_cancer/Task101_Colon/data_samples/labels/total"
output_dir = "/data/colon_cancer/Task101_Colon/data_samples/labels/colon"

# Option 1: Keep specific labels with their original values
keep_labels = [20]  

# Option 2: Keep specific labels and map them to new values
target_mapping = {
     20: 2,  
     75: 3,  
     76: 3,  
     77: 3,  
     78: 3, 
 }


# Run the function
keep_specific_labels(input_dir, output_dir, keep_labels=keep_labels, target_mapping=target_mapping)

In [ ]:
import os
import nibabel as nib
import numpy as np
from pathlib import Path

def merge_masks_from_folders(folder1, folder2, output_folder, 
                            folder1_labels=None, folder2_labels=None,
                            folder1_mapping=None, folder2_mapping=None,
                            conflict_strategy='max'):
    """
    Merges masks from two folders into single masks. Assumes corresponding files have the same names.
    
    Parameters:
    -----------
    folder1, folder2 : str
        Paths to folders containing input mask files
    output_folder : str
        Path to folder where merged masks will be saved
    folder1_labels, folder2_labels : list, optional
        Specific labels to keep from each folder. If None, keep all non-zero.
    folder1_mapping, folder2_mapping : dict, optional
        Mapping dictionaries to remap labels before merging
    conflict_strategy : str
        How to handle voxels where both masks have non-zero values:
        - 'max': Take the maximum value (default)
        - 'min': Take the minimum value
        - 'sum': Add the values
        - 'folder1': Prioritize folder1
        - 'folder2': Prioritize folder2
    """
    # Create output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Get files from both folders
    files1 = {f for f in os.listdir(folder1) if f.endswith('.nii.gz')}
    files2 = {f for f in os.listdir(folder2) if f.endswith('.nii.gz')}
    
    # Find common files (same filename exists in both folders)
    common_files = files1.intersection(files2)
    
    if not common_files:
        print("No matching .nii.gz files found in both folders.")
        return
    
    print(f"Found {len(common_files)} matching files to merge.")
    
    for filename in common_files:
        # Load first mask
        path1 = os.path.join(folder1, filename)
        img1 = nib.load(path1)
        data1 = img1.get_fdata()
        
        # Load second mask
        path2 = os.path.join(folder2, filename)
        img2 = nib.load(path2)
        data2 = img2.get_fdata()
        
        # Verify dimensions match
        if data1.shape != data2.shape:
            print(f"WARNING: Shape mismatch for {filename}. Skipping.")
            print(f"  {folder1}: {data1.shape}")
            print(f"  {folder2}: {data2.shape}")
            continue
        
        # Process first mask (keep only specified labels and apply mapping)
        processed1 = np.zeros_like(data1, dtype=np.uint16)
        
        if folder1_labels is None:
            # Keep all non-zero values
            mask1_nonzero = data1 > 0
            if folder1_mapping:
                # Apply mapping to all non-zero values
                for orig_val, new_val in folder1_mapping.items():
                    processed1[data1 == orig_val] = new_val
            else:
                processed1[mask1_nonzero] = data1[mask1_nonzero]
        else:
            # Keep only specified labels
            for label in folder1_labels:
                if folder1_mapping and label in folder1_mapping:
                    processed1[data1 == label] = folder1_mapping[label]
                else:
                    processed1[data1 == label] = label
        
        # Process second mask (keep only specified labels and apply mapping)
        processed2 = np.zeros_like(data2, dtype=np.uint16)
        
        if folder2_labels is None:
            # Keep all non-zero values
            mask2_nonzero = data2 > 0
            if folder2_mapping:
                # Apply mapping to all non-zero values
                for orig_val, new_val in folder2_mapping.items():
                    processed2[data2 == orig_val] = new_val
            else:
                processed2[mask2_nonzero] = data2[mask2_nonzero]
        else:
            # Keep only specified labels
            for label in folder2_labels:
                if folder2_mapping and label in folder2_mapping:
                    processed2[data2 == label] = folder2_mapping[label]
                else:
                    processed2[data2 == label] = label
        
        # Merge the two processed masks
        merged_data = np.zeros_like(processed1, dtype=np.uint16)
        
        # Find where each mask has non-zero values
        mask1_nonzero = processed1 > 0
        mask2_nonzero = processed2 > 0
        
        # Areas where only mask1 has values
        only_mask1 = mask1_nonzero & ~mask2_nonzero
        merged_data[only_mask1] = processed1[only_mask1]
        
        # Areas where only mask2 has values
        only_mask2 = ~mask1_nonzero & mask2_nonzero
        merged_data[only_mask2] = processed2[only_mask2]
        
        # Areas where both have values (conflicts)
        both = mask1_nonzero & mask2_nonzero
        
        if np.any(both):
            if conflict_strategy == 'max':
                merged_data[both] = np.maximum(processed1[both], processed2[both])
            elif conflict_strategy == 'min':
                merged_data[both] = np.minimum(processed1[both], processed2[both])
            elif conflict_strategy == 'sum':
                merged_data[both] = processed1[both] + processed2[both]
            elif conflict_strategy == 'folder1':
                merged_data[both] = processed1[both]
            elif conflict_strategy == 'folder2':
                merged_data[both] = processed2[both]
            else:
                # Default to max
                merged_data[both] = np.maximum(processed1[both], processed2[both])
            
            # Report conflicts
            conflict_count = np.sum(both)
            if conflict_count > 0:
                print(f"  {filename}: Resolved {conflict_count} conflicting voxels using '{conflict_strategy}' strategy")
        
        # Create new NIfTI image with original affine from first image
        new_img = nib.Nifti1Image(merged_data, img1.affine, img1.header)
        
        # Save merged mask
        output_path = os.path.join(output_folder, filename)
        nib.save(new_img, output_path)
        print(f"Merged: {filename}")

# --- Configuration ---
folder1 = '/data/colon_cancer/Task101_Colon/data_samples/labels_bwt'
folder2 = '/data/colon_cancer/Task101_Colon/data_samples/labels/colon+pelvis'
output_dir = '/data/colon_cancer/Task101_Colon/data_samples/labels/colon+pelvis_bwt_merged'

# Example 1: Simple merge - keep all non-zero values from both folders
# merge_masks_from_folders(folder1, folder2, output_dir)

# Example 2: Keep specific labels from each folder
# folder1_labels = [1, 2, 3]  # Only keep these labels from folder1
# folder2_labels = [4, 5, 6]  # Only keep these labels from folder2
# merge_masks_from_folders(folder1, folder2, output_dir, 
#                         folder1_labels=folder1_labels, 
#                         folder2_labels=folder2_labels)

# Example 3: Map labels from different sources to consistent values
# folder1_labels = [1, 2, 3]  # Original labels in folder1
# folder2_labels = [10, 20, 30]  # Original labels in folder2
# folder1_mapping = {1: 1, 2: 2, 3: 3}  # Keep folder1 labels as is
# folder2_mapping = {10: 1, 20: 2, 30: 4}  # Map folder2 labels: 10->1, 20->2, 30->4
# 
# merge_masks_from_folders(folder1, folder2, output_dir,
#                         folder1_labels=folder1_labels,
#                         folder2_labels=folder2_labels,
#                         folder1_mapping=folder1_mapping,
#                         folder2_mapping=folder2_mapping,
#                         conflict_strategy='max')

# Example 4: Merge two sets of labels with different naming conventions
folder1_labels = [1]  # e.g., left lung and right lung
folder2_labels = [2,3]  # e.g., heart and liver
# 
merge_masks_from_folders(folder1, folder2, output_dir,
                         folder1_labels=folder1_labels,
                         folder2_labels=folder2_labels,
                         conflict_strategy='folder1')

In [ ]:
import os
from pathlib import Path

def quick_add_prefix(input_folder, prefix="_0000"):
    """
    Quick function to add prefix to all .nii.gz files in a folder.
    """
    for filename in os.listdir(input_folder):
        if filename.endswith('.nii.gz'):
            # Split at the extension
            base = filename[:-7]  # removes '.nii.gz'
            new_name = f"{base}{prefix}.nii.gz"
            
            # Rename the file
            os.rename(
                os.path.join(input_folder, filename),
                os.path.join(input_folder, new_name)
            )
            print(f"Renamed: {filename} -> {new_name}")

# Usage
input_directory = '/data/colon_cancer/Task101_Colon/data_samples/labels/total'
quick_add_prefix(input_directory)